# Lab 2: Interpretable Models (Transparency by Design)

**Duration:** ~15 minutes  

---

## What you will learn

1. Understand the difference between **post-hoc explainability** (Lab 1) and **inherent interpretability** (this lab).
2. Train a **Logistic Regression** model and read its decision directly from the coefficients.
3. Train a **Decision Tree** and trace every prediction back to a human-readable rule.
4. **Compare** interpretable and black-box models on accuracy, and challenge the myth that interpretability always means worse performance.

---

## Inherent interpretability vs post-hoc explainability

| Approach | How it works | Example models |
|---|---|---|
| **Inherent interpretability** | The model *is* the explanation — no extra tools needed | Logistic Regression, Decision Tree, Linear Regression |
| **Post-hoc explainability** | A separate tool approximates or attributes the black-box decision | SHAP, LIME (Lab 1) |

&nbsp;

> **Key insight:** With an interpretable model you never have to ask *"why did the model do that?"* — the model's structure directly encodes the reasoning.  
> With a black-box model and post-hoc tools, you get an *approximation* of the reasoning — which may itself be wrong (as you saw in Lab 1).

We use the same **Breast Cancer Wisconsin** dataset as Lab 1 so results are directly comparable.

## Initial Setup

In [ ]:
# Run once to install any missing packages
import subprocess, sys
packages = ["scikit-learn", "matplotlib", "numpy", "pandas", "seaborn"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_").split(">")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages available.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
SEED = 42
print("Imports OK.")

---
## 1. Load the Data

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Scale features — important for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Classes: {dict(zip(data.target_names, np.bincount(y)))}")


---

## 2. Logistic Regression: a Linear Interpretable Model

### 2a. What exactly does Logistic Regression calculate?

Logistic Regression models the **log-odds** of the positive class as a linear combination of features:

$$
\log\frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_n x_n
$$

Each weight $w_i$ directly tells you:
- **Direction**: positive → feature pushes toward *benign*; negative → pushes toward *malignant*
- **Magnitude**: how strongly the feature influences the outcome (on the log-odds scale)

Unlike black-box models, there is **nothing hidden**: the model *is* its coefficients.


### 2b. Train & evaluate

In [ ]:
lr = LogisticRegression(max_iter=10_000, random_state=SEED)
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)
print(f"Logistic Regression test accuracy: {accuracy_score(y_test, y_pred_lr):.3f}\n")
print(classification_report(y_test, y_pred_lr, target_names=data.target_names))

### 2c. Inspect the coefficients (this IS the explanation)

In [ ]:
# The model coefficients are the full explanation — no SHAP/LIME needed
coef_df = pd.DataFrame({
    'feature':     data.feature_names,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

top_n = 15
plot_df = coef_df.head(top_n).sort_values('coefficient')
colours = ['#e74c3c' if c > 0 else '#2980b9' for c in plot_df['coefficient']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(plot_df['feature'], plot_df['coefficient'],
               color=colours, edgecolor='white', height=0.7)
ax.axvline(0, color='black', lw=0.8)

for bar, val in zip(bars, plot_df['coefficient']):
    sign = '+' if val > 0 else ''
    ax.text(val + (0.05 if val > 0 else -0.05),
            bar.get_y() + bar.get_height() / 2,
            f'{sign}{val:.2f}', va='center',
            ha='left' if val > 0 else 'right', fontsize=8)

red_p  = mpatches.Patch(color='#e74c3c', label='Positive → pushes toward benign')
blue_p = mpatches.Patch(color='#2980b9', label='Negative → pushes toward malignant')
ax.legend(handles=[red_p, blue_p], fontsize=9)

ax.set_xlabel("Coefficient value (standardised features)")
ax.set_title(
    f"Logistic Regression — top {top_n} feature coefficients\n"
    "(these ARE the explanation — no post-hoc tools needed)",
    fontsize=11, fontweight='bold'
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Top 5 features (by absolute coefficient):")
display(coef_df.head(5).reset_index(drop=True))

### 2d. Exercises

1. Which feature has the **largest positive** coefficient? What does that mean clinically?
2. Compare the top features here to the SHAP global importance from Lab 1. Do they agree?
3. The coefficients are on **standardised** features. Why is standardisation important for interpreting logistic regression coefficients?
4. **(Stretch)** Logistic Regression has a single global explanation. Can it explain an *individual* prediction in the same way SHAP's waterfall can? How would you compute the contribution of each feature for one specific sample?


---

## 3. Decision Tree: Rules You Can Read and Trace

### 3a. What exactly does a Decision Tree calculate?

A Decision Tree partitions the feature space using a series of **axis-aligned threshold rules**:  

> *If `mean concave points ≤ 0.051` → go left, else → go right.*

At each node the algorithm picks the feature and threshold that maximises **information gain** (or equivalently minimises **Gini impurity**):

$$
\text{Gini}(t) = 1 - \sum_{k} p_k^2
$$

where $p_k$ is the fraction of class $k$ samples at node $t$.  
The best split minimises the weighted average Gini of the two child nodes.

Every prediction can be traced back to a root-to-leaf path, which is simply a list of plain-English rules.


### 3b. Train a full Decision Tree on all features

In [ ]:
# Shallow tree for interpretability: max_depth limits the number of rules
dt = DecisionTreeClassifier(max_depth=4, random_state=SEED)
dt.fit(X_train, y_train)          # no scaling needed for trees

y_pred_dt = dt.predict(X_test)
print(f"Decision Tree (depth 4) test accuracy: {accuracy_score(y_test, y_pred_dt):.3f}\n")
print(classification_report(y_test, y_pred_dt, target_names=data.target_names))

### 3c. Visualise the full tree

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt,
    feature_names=list(data.feature_names),
    class_names=list(data.target_names),
    filled=True, rounded=True, fontsize=15,
    impurity=True, proportion=False,
    ax=ax
)
ax.set_title(
    "Full Decision Tree (max_depth=4) — every split is a human-readable rule",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

### 3d. Extract the decision rule for one prediction

In [ ]:
# Print the full text rule set for the tree
rules_text = export_text(dt, feature_names=list(data.feature_names))
# Show only the first ~60 lines to keep output manageable
print("\n".join(rules_text.split("\n")[:60]))
print("... (truncated)")

In [ ]:
# Trace the decision path for one specific test sample
SAMPLE_IDX = 5
sample = X_test.iloc[[SAMPLE_IDX]]
true_label = data.target_names[y_test.iloc[SAMPLE_IDX]]
pred_label = data.target_names[dt.predict(sample)[0]]

print(f"Sample {SAMPLE_IDX}: true = {true_label}, predicted = {pred_label}\n")

# Walk the decision path
node_indicator  = dt.decision_path(sample)
leaf_id         = dt.apply(sample)[0]
feature_idx     = dt.tree_.feature
threshold_arr   = dt.tree_.threshold
feat_names      = list(data.feature_names)

print("Decision path (root → leaf):")
print("-" * 55)
for node_id in node_indicator.indices:
    if node_id == leaf_id:
        proportions = dt.tree_.value[node_id][0]
        class_counts = np.round(proportions * dt.tree_.n_node_samples[node_id]).astype(int)
        pred_class   = data.target_names[np.argmax(proportions)]
        print(f"  LEAF → predict '{pred_class}'  "
              f"(node samples: malignant={class_counts[0]}, benign={class_counts[1]})")
    else:
        feat    = feat_names[feature_idx[node_id]]
        thresh  = threshold_arr[node_id]
        value   = sample[feat].values[0]
        direction = "≤" if value <= thresh else ">"
        print(f"  Node {node_id:3d}: {feat} = {value:.4f}  {direction}  {thresh:.4f}")
print("-" * 55)

### 3e. Feature importance from the tree

In [ ]:
importance_df = pd.DataFrame({
    'feature':    data.feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

top_n = 15
plot_df = importance_df.head(top_n).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df['feature'], plot_df['importance'],
        color='#27ae60', edgecolor='white', height=0.7)
ax.set_xlabel("Gini importance (mean decrease in impurity)")
ax.set_title(
    f"Decision Tree - top {top_n} features by Gini importance",
    fontsize=11, fontweight='bold'
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 3f. Exercises

1. Look at the decision path printed above. Can you explain in plain English *why* the model predicted what it did for sample 5?
2. The tree uses `max_depth=4`. Try `max_depth=2` and `max_depth=8`. How does accuracy change?  
   What is the trade-off between depth, accuracy, and interpretability?
3. Compare the tree's top features to the Logistic Regression coefficients. Do they agree on the most important features?


---

## 4. The Accuracy–Interpretability Trade-off Myth

A common belief is that interpretable models are less accurate.  
This is sometimes true, but not always. Let us test it directly.

In [ ]:
# Train all three models and collect accuracy
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train, y_train)

results = [
    ("Logistic Regression",   "Inherently interpretable",  accuracy_score(y_test, lr.predict(X_test_sc))),
    ("Decision Tree (d=4)",   "Inherently interpretable",  accuracy_score(y_test, y_pred_dt)),
    ("Random Forest (100T)",  "Black-box (needs SHAP/LIME)", accuracy_score(y_test, rf.predict(X_test))),
]

results_df = pd.DataFrame(results, columns=['Model', 'Transparency', 'Test Accuracy'])
display(results_df)

# Bar chart
colours_bar = {'Inherently interpretable': '#27ae60',
               'Black-box (needs SHAP/LIME)': '#e74c3c'}
bar_colours = [colours_bar[t] for t in results_df['Transparency']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(results_df['Model'], results_df['Test Accuracy'],
              color=bar_colours, edgecolor='white', width=0.5)

for bar, val in zip(bars, results_df['Test Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0.85, 1.02)
ax.set_ylabel("Test Accuracy")
ax.set_title("Accuracy comparison — interpretable vs black-box models\n"
             "(on Breast Cancer Wisconsin dataset)",
             fontsize=11, fontweight='bold')

green_p = mpatches.Patch(color='#27ae60', label='Inherently interpretable')
red_p   = mpatches.Patch(color='#e74c3c', label='Black-box (needs SHAP/LIME)')
ax.legend(handles=[green_p, red_p], fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 4a. When interpretability comes at zero cost

On this dataset the Logistic Regression already achieves competitive accuracy with no interpretability cost at all.  
Before reaching for a complex black-box model (and then spending effort on SHAP/LIME), always ask:

> *Does a simpler, inherently interpretable model already solve the problem well enough?*

If yes, choose it. You get:
- **Free explanations**: no post-hoc tools, no approximations
- **Deterministic** explanations: the same input always gives the same explanation
- **Regulatory defensibility**: easier to audit and validate
- **Simpler deployment**: one model to maintain, not model + explainer


---

## 5. How interpretable are my models?

We have compared accuracy, but how do we compare *interpretability* objectively?  
This exercise introduces two model-agnostic measures from [Towards Objective Metrics for Explanation Quality (CEUR 2023)](https://ceur-ws.org/Vol-3554/paper4.pdf).

### Compactness

Compactness measures how concisely a model explains a prediction: fewer features used = more compact.

$$\text{Explanation compactness} = 1 - \frac{|F_{ex}| - 1}{|F|}$$

where $|F|$ = total features in the model and $|F_{ex}|$ = features used in this specific explanation.  
Subtracting 1 avoids penalising single-feature explanations. Higher = more compact (range 0–1).

**Model compactness** is the mean over all individual explanations:

$$\text{Model compactness} = \frac{1}{|E|} \sum_{ex \in E} \left(1 - \frac{|F_{ex}| - 1}{|F|}\right)$$

**For decision trees:** $|F_{ex}|$ = number of unique features on the root-to-leaf path for that instance.  
**For logistic regression:** feature importances are derived from per-instance contributions $|w_j \cdot x_{ij}|$ (analogous to t-statistics on standardised features). Softmax is applied to get a probability distribution; features are selected until their cumulative mass reaches a threshold.

### Stability

A model is considered stable if similar instances receive similar explanations.

$$\text{stability} = \frac{|I_s \cap E_s|}{|I_s|}$$

| Symbol | Meaning |
|--------|---------|
| $I_s$  | **Instance space**: test neighbours within radius $T_r$ (feature space) |
| $E_s$  | **Explanation space**: instances sharing the same leaf/rule (DT) or same top-feature set (LR) |

**Radius threshold $T_r$** is computed from the training set using $k$-NN ($k=9$):

$$T_r = \frac{1}{N} \sum_{i=0}^{N} t_i$$

where $t_i$ is the distance from instance $i$ to its $k$-th nearest neighbour.

**Exercise:** Run the cells below to train 2 decision trees (depth 2 and depth 8) and 2 logistic regressions (C=1.0 and C=0.01). Then examine the computed compactness and stability scores.

In [ ]:
from scipy.special import softmax as sp_softmax
from sklearn.neighbors import NearestNeighbors

# --- Two decision trees with different depths --------------------------
dt_shallow = DecisionTreeClassifier(max_depth=4, random_state=SEED)
dt_deep    = DecisionTreeClassifier(max_depth=8, random_state=SEED)
dt_shallow.fit(X_train, y_train)
dt_deep.fit(X_train, y_train)

# --- Two logistic regressions with different regularisation ------------
# C = inverse regularisation strength: low C = strong regularisation
lr_default = LogisticRegression(C=1.0,  max_iter=10_000, random_state=SEED)
lr_strong  = LogisticRegression(C=0.1, max_iter=10_000, random_state=SEED)
lr_default.fit(X_train_sc, y_train)
lr_strong.fit(X_train_sc, y_train)

models = {
    'DT (depth=4)': ('dt', dt_shallow),
    'DT (depth=8)': ('dt', dt_deep),
    'LR (C=1.0)':   ('lr', lr_default),
    'LR (C=0.1)':  ('lr', lr_strong),
}

# Convert to arrays once for reuse in distance calculations
X_train_arr = X_train.values
X_test_arr  = X_test.values

print(f"{'Model':<20}  {'Accuracy':>8}")
print("-" * 32)
for name, (kind, model) in models.items():
    test_data = X_test if kind == 'dt' else X_test_sc
    acc = accuracy_score(y_test, model.predict(test_data))
    print(f"{name:<20}  {acc:>8.3f}")

In [ ]:
# === 5a. Compactness ===================================================
N_FEATURES        = X_train.shape[1]  # 30 features in the breast-cancer dataset
SOFTMAX_THRESHOLD = 0.50              # cumulative probability mass threshold for LR


def dt_fex_sizes(tree, X):
    """
    For each sample in X, count the number of *unique* features on the
    path from the root to the leaf.  This is |Fex| for one explanation.
    """
    X_arr     = X.values if isinstance(X, pd.DataFrame) else X
    indicator = tree.decision_path(X_arr)
    feat_idx  = tree.tree_.feature          # feature used at each node (-2 = leaf)
    sizes = []
    for i in range(X_arr.shape[0]):
        nodes = indicator.getrow(i).indices
        # exclude leaf nodes (feature == -2 means no split)
        feats = {feat_idx[n] for n in nodes if feat_idx[n] != -2}
        sizes.append(max(1, len(feats)))    # at least 1 to avoid division edge-cases
    return sizes


def lr_fex_sizes(lr_model, X_sc, threshold=SOFTMAX_THRESHOLD):
    """
    Per-instance |Fex| for logistic regression.

    Feature importance for instance i:  |coef_j * x_{ij}|
    (the contribution of feature j to the log-odds for this specific instance —
    analogous to a per-instance t-statistic on standardised features).

    Softmax converts these to a probability distribution; features are
    selected in descending order until their cumulative mass >= threshold.
    """
    contributions = np.abs(lr_model.coef_[0] * X_sc)   # shape (n_samples, n_features)
    sizes = []
    for imp in contributions:
        probs  = sp_softmax(imp)
        order  = np.argsort(probs)[::-1]
        cumsum, count = 0.0, 0
        for idx in order:
            cumsum += probs[idx]
            count  += 1
            if cumsum >= threshold:
                break
        sizes.append(max(1, count))
    return sizes


def model_compactness(fex_sizes, n_features=N_FEATURES):
    """Mean explanation compactness over all instances (eq. 2 in the paper)."""
    return float(np.mean([1 - (f - 1) / n_features for f in fex_sizes]))


# Compute compactness for each model
compactness_results = {}
print(f"{'Model':<20}  {'Compactness':>11}  {'Mean |Fex|':>10}")
print("-" * 46)
for name, (kind, model) in models.items():
    fex = (dt_fex_sizes(model, X_test)
           if kind == 'dt'
           else lr_fex_sizes(model, X_test_sc))
    compactness_results[name] = model_compactness(fex)
    print(f"{name:<20}  {compactness_results[name]:>11.4f}  {np.mean(fex):>10.2f}")

In [ ]:
# === 5b. Stability =====================================================
# Reference implementation: autoxplain/knn.py + autoxplain/models/score_lr.py
K = 9   # number of nearest neighbours (as used in the paper)


def knn_radius_threshold(X_arr, k=K):
    """
    Replicates knn.py: fit k-NN on training data, compute max distance to k-th
    neighbour per instance, return mean of those maxima as the radius threshold.
    Note: sklearn includes self when querying training points, so with n_neighbors=k
    the farthest of the k returned is the (k-1)-th true neighbour.
    """
    nn = NearestNeighbors(n_neighbors=k, algorithm="ball_tree").fit(X_arr)
    dists, _ = nn.kneighbors(X_arr)          # shape (n, k), first col often ~0 (self)
    return float(np.mean(dists.max(axis=1))) # mean of per-instance max distances


def radius_neighbors_test(X_train_arr, X_test_arr, Tr):
    """Build radius index on test set, query test set — mirrors knn.py neighbors()."""
    nn = NearestNeighbors(radius=Tr, algorithm="ball_tree").fit(X_test_arr)
    return nn.radius_neighbors(X_test_arr, return_distance=False)


def stability_dt(tree, X_train_arr, X_test_arr):
    """
    Stability for a decision tree.
    Instance space  (Is): radius neighbours in feature space (threshold from training).
    Explanation space (Es): instances sharing the same leaf node.
    stability = |Is ∩ Es| / |Is|
    """
    Tr       = knn_radius_threshold(X_train_arr)
    Is_all   = radius_neighbors_test(X_train_arr, X_test_arr, Tr)
    leaf_ids = tree.apply(X_test_arr)

    scores = []
    for i, Is_raw in enumerate(Is_all):
        Is = set(Is_raw)                # keep self (matches reference denominator)
        if len(Is) == 0:
            continue
        Es = set(np.where(leaf_ids == leaf_ids[i])[0])
        scores.append(len(Is & Es) / len(Is))
    return float(np.mean(scores)) if scores else 0.0


def stability_lr(lr_model, X_train_sc, X_test_sc):
    """
    Stability for logistic regression — mirrors score_lr.py.

    Instance space (Is): radius neighbours in *feature value* space.
    Explanation space (Es): radius neighbours in *mean-importance scalar* space.
        Importance per instance = softmax(|coef_j * x_ij|) — analogous to the
        t-statistic approach in the paper; reduced to a single scalar via mean.

    stability = |Is ∩ Es| / |Is|
    """
    # --- Instance space: knn.py neighbors() replicated exactly ---
    Tr_inst   = knn_radius_threshold(X_train_sc)
    data_labels = radius_neighbors_test(X_train_sc, X_test_sc, Tr_inst)

    # --- Explanation space: mean-importance scalar per instance ---
    # Per-instance feature importance: softmax of |coef_j * x_ij|
    imp_train = sp_softmax(np.abs(lr_model.coef_[0] * X_train_sc), axis=1)
    imp_test  = sp_softmax(np.abs(lr_model.coef_[0] * X_test_sc),  axis=1)

    # Collapse to a single scalar per instance (mean probability mass)
    scalar_train = np.mean(imp_train, axis=1).reshape(-1, 1)
    scalar_test  = np.mean(imp_test,  axis=1).reshape(-1, 1)

    Tr_exp    = knn_radius_threshold(scalar_train)
    exp_indices = radius_neighbors_test(scalar_train, scalar_test, Tr_exp)

    # --- stability = |Is ∩ Es| / |Is| ---
    scores = []
    for i in range(len(X_test_sc)):
        Is = set(data_labels[i])       # keep self, matches reference denominator
        if len(Is) == 0:
            continue
        Es = set(exp_indices[i])
        scores.append(len(Is & Es) / len(Is))
    return float(np.mean(scores)) if scores else 0.0


# Compute stability for each model
stability_results = {}
print(f"{'Model':<20}  {'Stability':>9}  {'Tr_inst (radius)':>16}")
print("-" * 50)
for name, (kind, model) in models.items():
    if kind == 'dt':
        Tr  = knn_radius_threshold(X_train_arr)
        stab = stability_dt(model, X_train_arr, X_test_arr)
    else:
        Tr   = knn_radius_threshold(X_train_sc)
        stab = stability_lr(model, X_train_sc, X_test_sc)
    stability_results[name] = stab
    print(f"{name:<20}  {stab:>9.4f}  {Tr:>16.4f}")

In [ ]:
# === 5c. Summary — Accuracy · Compactness · Stability =================
summary_rows = []
for name, (kind, model) in models.items():
    test_data = X_test if kind == 'dt' else X_test_sc
    acc = accuracy_score(y_test, model.predict(test_data))
    summary_rows.append({
        'Model':       name,
        'Accuracy':    round(acc, 3),
        'Compactness': round(compactness_results[name], 3),
        'Stability':   round(stability_results[name], 3),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Grouped bar chart — one panel per metric
palette  = ['#3498db', '#2471a3', '#e74c3c', '#c0392b']   # blue = DT, red = LR
metrics  = ['Accuracy', 'Compactness', 'Stability']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric in zip(axes, metrics):
    bars = ax.bar(summary_df['Model'], summary_df[metric],
                  color=palette, edgecolor='white', width=0.55)
    for bar, val in zip(bars, summary_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.18)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Colour legend
dt_patch = mpatches.Patch(color='#3498db', label='Decision Tree')
lr_patch = mpatches.Patch(color='#e74c3c', label='Logistic Regression')
fig.legend(handles=[dt_patch, lr_patch], loc='upper right',
           bbox_to_anchor=(1.0, 1.02), fontsize=9)

fig.suptitle(
    'Accuracy · Compactness · Stability comparison\n'
    '(Breast Cancer Wisconsin dataset)',
    fontsize=12, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.show()

**Discussion questions:**
1. Which model is most compact? How does tree depth affect compactness?
2. Does higher regularisation (lower C) in LR change compactness? Why?
3. Which model is most stable? Is there a depth–stability trade-off for decision trees?
4. Can you find a single model that balances all three properties (accuracy, compactness, and stability)? Which model would you choose for this medical dataset?